# azure-ai-agentserver-core smoke test

Exercises the public API surface of `azure.ai.agentserver.core` (v2.0.0b5):
`AgentConfig`, `AgentServerHost`, `create_error_response`, tracing helpers,
and the middleware classes.

In [ ]:
# Example 1: Public API surface — every name in __all__ resolves on the package.
import azure.ai.agentserver.core as aas_core

names = aas_core.__all__
print(f"__all__ has {len(names)} names")
for name in names:
    obj = getattr(aas_core, name)
    print(f"  {name:40s} {type(obj).__name__}")
    assert obj is not None

expected = {
    "AgentConfig", "AgentServerHost", "InboundRequestLoggingMiddleware",
    "RequestIdMiddleware", "build_server_version", "configure_observability",
    "create_error_response", "detach_context", "end_span", "flush_spans",
    "record_error", "set_current_span", "trace_stream",
}
assert expected.issubset(set(names)), expected - set(names)

In [ ]:
# Example 2: AgentConfig.from_env() resolves env vars + defaults.
import os

from azure.ai.agentserver.core import AgentConfig

for key in ("FOUNDRY_AGENT_NAME", "FOUNDRY_AGENT_VERSION", "PORT"):
    os.environ.pop(key, None)

os.environ["FOUNDRY_AGENT_NAME"] = "smoke-test-agent"
os.environ["FOUNDRY_AGENT_VERSION"] = "0.1.0"
os.environ["PORT"] = "9090"

cfg = AgentConfig.from_env()
print(f"agent_name    = {cfg.agent_name!r}")
print(f"agent_version = {cfg.agent_version!r}")
print(f"agent_id      = {cfg.agent_id!r}")
print(f"port          = {cfg.port}")
print(f"is_hosted     = {cfg.is_hosted}")

assert cfg.agent_name == "smoke-test-agent"
assert cfg.agent_version == "0.1.0"
assert cfg.port == 9090

In [ ]:
# Example 3: create_error_response builds a Starlette JSONResponse with the standard envelope.
import json

from azure.ai.agentserver.core import create_error_response
from starlette.responses import JSONResponse

resp = create_error_response(
    "not_found",
    "Resource missing",
    status_code=404,
    error_type="client_error",
    details=[{"code": "missing_id", "message": "id is required"}],
)

assert isinstance(resp, JSONResponse)
assert resp.status_code == 404

body = json.loads(resp.body)
print(json.dumps(body, indent=2))

assert body["error"]["code"] == "not_found"
assert body["error"]["type"] == "client_error"
assert body["error"]["details"][0]["code"] == "missing_id"

In [ ]:
# Example 4: AgentServerHost is a Starlette app with health probe + platform-server header.
from azure.ai.agentserver.core import AgentServerHost
from starlette.applications import Starlette
from starlette.testclient import TestClient

# Skip observability setup so we don't wire OTel exporters in-test.
app = AgentServerHost(configure_observability=None, log_level="INFO")

assert isinstance(app, Starlette)
assert app.config.port > 0

with TestClient(app) as client:
    r = client.get("/readiness")
    print(f"GET /readiness -> {r.status_code}")
    print(f"x-platform-server: {r.headers.get('x-platform-server')!r}")
    assert r.status_code == 200
    assert "azure-ai-agentserver-core" in (r.headers.get("x-platform-server") or "")

In [ ]:
# Example 5: tracing helpers compose with OpenTelemetry — end_span / record_error / flush_spans.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from azure.ai.agentserver.core import end_span, flush_spans, record_error

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

span = trace.get_tracer("smoke-test").start_span("unit-of-work")
try:
    raise ValueError("boom")
except ValueError as exc:
    record_error(span, exc)
    end_span(span, exc)

flush_spans(timeout_millis=2000)
spans = exporter.get_finished_spans()
print(f"spans={len(spans)} name={spans[0].name!r} events={[e.name for e in spans[0].events]}")
assert len(spans) == 1 and spans[0].name == "unit-of-work"
assert any(e.name == "exception" for e in spans[0].events)